# CubiCasa5K + CubiGraph5K — local macOS runner

Run this notebook from the `FYP` folder on your Mac. It replaces Colab uploads, `/content` paths, shell magics, forced CUDA, and browser downloads with local files.

**Important distinction:** CubiCasa5K parses a raster image into polygons and segmentation masks. CubiGraph5K then derives a relation graph from a semantically annotated SVG; it is not an end-to-end raster-to-graph model. The SVG created by this notebook is an experimental bridge and should be visually checked.


## Before you run

In Terminal:

```bash
cd ~/Downloads/FYP
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements_macos.txt
jupyter lab
```

Put one plan image in `input/` and set `INPUT_IMAGE` below. The first setup run clones the public repositories and downloads the published CubiCasa5K checkpoint (~203 MB), so it needs internet access.


In [ ]:
from pathlib import Path
import os

# Start Jupyter from ~/Downloads/FYP so this resolves to the notebook project folder.
PROJECT_ROOT = Path.cwd().resolve()
INPUT_DIR = PROJECT_ROOT / 'input'
WORKDIR = PROJECT_ROOT / 'floorplan_parser_output'
THIRD_PARTY = PROJECT_ROOT / 'third_party'
CUBICASA_REPO = THIRD_PARTY / 'CubiCasa5k'
CUBIGRAPH_REPO = THIRD_PARTY / 'CubiGraph5K'
CHECKPOINT = CUBICASA_REPO / 'model_best_val_loss_var.pkl'

for folder in (INPUT_DIR, WORKDIR, THIRD_PARTY):
    folder.mkdir(parents=True, exist_ok=True)

# Change this filename only. PNG/JPG/JPEG are supported.
INPUT_IMAGE = INPUT_DIR / 'floorplan.png'
print('Project root:', PROJECT_ROOT)
print('Expected input:', INPUT_IMAGE)
print('Outputs:', WORKDIR)


## 1. Install dependencies, clone sources, and fetch the published checkpoint


In [ ]:
import subprocess
import sys

def run(command):
    print('+', ' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), check=True)

# Re-running is safe: pip only changes what is missing/outdated.
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', PROJECT_ROOT / 'requirements_macos.txt'])

if not CUBICASA_REPO.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/CubiCasa/CubiCasa5k.git', CUBICASA_REPO])
if not CUBIGRAPH_REPO.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/luyueheng/CubiGraph5K.git', CUBIGRAPH_REPO])

if not CHECKPOINT.exists():
    import gdown
    url = 'https://drive.google.com/uc?id=1gRB7ez1e4H7a9Y09lLqRuna0luZO5VRK'
    downloaded = gdown.download(url=url, output=str(CHECKPOINT), quiet=False)
    if not downloaded:
        raise RuntimeError('Checkpoint download failed. Check network access, then rerun this cell.')

print('CubiCasa5K:', CUBICASA_REPO)
print('CubiGraph5K:', CUBIGRAPH_REPO)
print('Checkpoint:', CHECKPOINT)


## 2. Load one local floor-plan image


In [ ]:
from IPython.display import display
from PIL import Image

if not INPUT_IMAGE.exists():
    raise FileNotFoundError(
        f'Put a PNG/JPG/JPEG floor plan at {INPUT_IMAGE}, or edit INPUT_IMAGE in the configuration cell.'
    )

source_image = Image.open(INPUT_IMAGE).convert('RGB')
display(source_image)
print('Loaded:', INPUT_IMAGE.name, '| original size:', source_image.size)


## 3. Load CubiCasa5K with a Mac-friendly device fallback


In [ ]:
import matplotlib
import matplotlib.cm as cm
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Compatibility patch for older CubiCasa5K code on recent Matplotlib.
# CubiCasa's discrete_cmap() may be called repeatedly as cells are rerun.
_original_register_cmap = cm.register_cmap if hasattr(cm, 'register_cmap') else None
def _safe_register_cmap(cmap=None, name=None, **kwargs):
    cmap_name = name or (cmap.name if hasattr(cmap, 'name') else None)
    try:
        if _original_register_cmap is not None:
            return _original_register_cmap(cmap=cmap, name=name, **kwargs)
        if cmap_name and hasattr(matplotlib, 'colormaps'):
            return matplotlib.colormaps.register(cmap=cmap, name=cmap_name)
    except ValueError as err:
        if 'already registered' not in str(err):
            raise

cm.register_cmap = _safe_register_cmap

sys.path.insert(0, str(CUBICASA_REPO))
from floortrans.models import get_model
from floortrans.loaders import RotateNTurns
from floortrans.plotting import discrete_cmap, polygons_to_image
from floortrans.post_prosessing import split_prediction, get_polygons

# Apple Silicon uses MPS when supported; Intel Macs and unsupported operations fall back to CPU.
if torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

split = [21, 12, 11]
room_classes = ['Background', 'Outdoor', 'Wall', 'Kitchen', 'Living Room', 'Bed Room', 'Bath', 'Entry', 'Railing', 'Storage', 'Garage', 'Undefined']
icon_classes = ['No Icon', 'Window', 'Door', 'Closet', 'Electrical Appliance', 'Toilet', 'Sink', 'Sauna Bench', 'Fire Place', 'Bathtub', 'Chimney']

# CubiCasa5K loads model_1427.pth via a relative path inside get_model().
# Temporarily entering its repository keeps that legacy loader working on macOS.
previous_cwd = Path.cwd()
os.chdir(CUBICASA_REPO)
try:
    model = get_model('hg_furukawa_original', 51)
finally:
    os.chdir(previous_cwd)
model.conv4_ = torch.nn.Conv2d(256, 44, bias=True, kernel_size=1)
model.upsample = torch.nn.ConvTranspose2d(44, 44, kernel_size=4, stride=4)
checkpoint = torch.load(CHECKPOINT, map_location='cpu')
model.load_state_dict(checkpoint['model_state'])
model.to(device).eval()
rot = RotateNTurns()
discrete_cmap()
print('Model loaded on', device)


## 4. Parse the image and save masks/polygons


In [ ]:
import scipy.stats as stats
import floortrans.post_prosessing

# Compatibility patch for SciPy >= 1.11, used by floortrans post-processing.
original_mode = stats.mode
def _patched_mode(*args, **kwargs):
    result = original_mode(*args, **kwargs)
    if hasattr(result, 'mode') and np.isscalar(result.mode):
        class LegacyMode:
            def __init__(self, value):
                self.mode = [value]
        return LegacyMode(result.mode)
    return result
floortrans.post_prosessing.stats.mode = _patched_mode

MAX_SIDE = 1024  # Increase only if your Mac has sufficient memory.
img = source_image.copy()
scale = min(1.0, MAX_SIDE / max(img.size))
resized_size = (round(img.width * scale), round(img.height * scale))
img = img.resize(resized_size, Image.Resampling.LANCZOS)
padded_w = (img.width + 31) // 32 * 32
padded_h = (img.height + 31) // 32 * 32
canvas = Image.new('RGB', (padded_w, padded_h), 'white')
canvas.paste(img, (0, 0))
x = (np.asarray(canvas).astype(np.float32) / 255.0 - 0.5) * 2.0
image_tensor = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0)

def run_inference(target_device):
    model.to(target_device).eval()
    tensor = image_tensor.to(target_device)
    with torch.no_grad():
        predictions = []
        for forward, backward in [(0, 0), (1, -1), (2, 2), (-1, 1)]:
            pred = model(rot(tensor, 'tensor', forward))
            pred = rot(pred, 'tensor', backward)
            pred = rot(pred, 'points', backward)
            pred = F.interpolate(pred, size=(padded_h, padded_w), mode='bilinear', align_corners=True)
            predictions.append(pred)
        return torch.stack(predictions).mean(0).cpu()

try:
    prediction = run_inference(device)
except RuntimeError as err:
    if device.type != 'mps':
        raise
    print('MPS inference failed; retrying on CPU:', err)
    device = torch.device('cpu')
    prediction = run_inference(device)

heatmaps, rooms, icons = split_prediction(prediction, (padded_h, padded_w), split)
polygons, types, room_polygons, room_types = get_polygons((heatmaps, rooms, icons), 0.2, [1, 2])
room_seg, icon_seg = polygons_to_image(polygons, types, room_polygons, room_types, padded_h, padded_w)

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
axes[0].imshow(canvas); axes[0].set_title('Processed input'); axes[0].axis('off')
axes[1].imshow(room_seg, cmap='rooms', vmin=0, vmax=10.9); axes[1].set_title('CubiCasa5K rooms / walls'); axes[1].axis('off')
axes[2].imshow(icon_seg, cmap='icons', vmin=0, vmax=10.9); axes[2].set_title('CubiCasa5K icons'); axes[2].axis('off')
plt.tight_layout()

np.savez_compressed(WORKDIR / 'cubicasa5k_prediction.npz', room_seg=room_seg, icon_seg=icon_seg)
print('Saved masks:', WORKDIR / 'cubicasa5k_prediction.npz')


## 5. Export an experimental CubiCasa-style SVG for CubiGraph5K


In [ ]:
import svgwrite
from shapely.geometry import Polygon, MultiPolygon

def save_as_cubicasa_svg(output_path, room_polys, room_classes_meta, icon_polys, icon_classes_meta, width, height):
    dwg = svgwrite.Drawing(output_path, size=(width, height), profile='full')

    def add_geom(group, geom, class_label):
        if isinstance(geom, Polygon):
            group.add(dwg.polygon(points=[(float(x), float(y)) for x, y in geom.exterior.coords], class_=class_label))
        elif isinstance(geom, MultiPolygon):
            for sub in geom.geoms:
                group.add(dwg.polygon(points=[(float(x), float(y)) for x, y in sub.exterior.coords], class_=class_label))

    # CubiGraph5K expects *each* room group to carry `Space <RoomType>`.
    # Do not wrap those groups in a parent `Space` group: its parser would read
    # that parent as an extra room, using the first nested polygon.
    cubigraph_room_name = {
        'Living Room': 'LivingRoom', 'Bed Room': 'Bedroom', 'Bath': 'Bath',
        'Entry': 'Entry', 'Kitchen': 'Kitchen', 'Storage': 'Storage',
        'Garage': 'Garage', 'Outdoor': 'Outdoor', 'Undefined': 'Undefined'
    }
    for geom, meta in zip(room_polys, room_classes_meta):
        # CubiGraph's parser reads only the first polygon in each Space group.
        # Split MultiPolygons into individual Space groups so no rooms vanish.
        parts = [geom] if isinstance(geom, Polygon) else list(geom.geoms) if isinstance(geom, MultiPolygon) else []
        raw_name = room_classes[meta['class']]
        class_name = cubigraph_room_name.get(raw_name, 'Undefined')
        for part in parts:
            if not part.is_empty and part.area > 1e-6:
                room_group = dwg.add(dwg.g(class_=f'Space {class_name}'))
                add_geom(room_group, part, class_name)

    # Each door is also a direct group. A parent Threshold group would become
    # a spurious extra door in CubiGraph5K's legacy SVG parser.
    for poly_data, meta in zip(icon_polys, icon_classes_meta):
        class_name = icon_classes[meta['class']]
        # CubiGraph calls every exported Threshold a Door. Keeping Windows
        # here would fabricate door-connected room pairs.
        if class_name == 'Door':
            item = dwg.add(dwg.g(class_='Threshold'))
            if isinstance(poly_data, np.ndarray):
                points = [(float(x), float(y)) for x, y in poly_data.reshape(-1, 2)]
                item.add(dwg.polygon(points=points, class_=class_name))
            else:
                add_geom(item, poly_data, class_name)
    dwg.save()

svg_path = WORKDIR / 'model.svg'
save_as_cubicasa_svg(svg_path, room_polygons, room_types, polygons, types, padded_w, padded_h)
print('Saved experimental SVG:', svg_path)


## 6. Generate a CubiGraph5K room-relation graph


In [ ]:
import json
import warnings
from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning

sys.path.insert(0, str(CUBIGRAPH_REPO / 'src'))
import plan
from plan import Plan, Room
from IPython.display import SVG, display

warnings.filterwarnings('ignore', category=XMLParsedAsHTMLWarning)

with open(svg_path, encoding='utf-8') as handle:
    soup = BeautifulSoup(handle, 'lxml')
plan_obj = Plan(soup.find('svg'))
# Experimental relation policy: a shared door takes precedence over generic adjacency.
# This avoids CubiGraph's legacy order, where an ordinary interior door is hidden
# by the rooms' buffered-boundary overlap.
plan_obj.relation = []
for room in plan_obj.rooms:
    room.get_adjacent_doors(plan_obj.doors)
for i, room1 in enumerate(plan_obj.rooms):
    for room2 in plan_obj.rooms[i + 1:]:
        if room1.adjacent_doors.intersection(room2.adjacent_doors):
            label = 2  # via-door
        elif room1.to_shapely_polygon().buffer(1.0).intersection(room2.to_shapely_polygon().buffer(1.0)).area > 5.0:
            label = 1  # adjacent only
        else:
            label = 0
        plan_obj.relation.append((room1.name, label, room2.name))
adjacency = plan_obj.get_adjacency_list()

graph_svg_path = WORKDIR / 'cubigraph_relations.svg'
graph_svg_path.write_text(str(plan_obj.generate_relation_svg()), encoding='utf-8')
graph_json_path = WORKDIR / 'cubigraph_adjacency.json'
graph_json_path.write_text(json.dumps(adjacency, indent=2), encoding='utf-8')

print(json.dumps(adjacency, indent=2))
display(SVG(filename=str(graph_svg_path)))
print('Saved graph visualisation:', graph_svg_path)
print('Saved adjacency JSON:', graph_json_path)


## 7. Package the outputs locally


In [ ]:
import shutil
archive = shutil.make_archive(str(PROJECT_ROOT / 'floorplan_parser_output'), 'zip', WORKDIR)
print('Created:', archive)
print('You can inspect all outputs in:', WORKDIR)
